In [0]:
# ==========================================
# 05. INGESTA BRONZE - EMPLEADOS
# Proyecto: ElectroCasa
# Tecnología: Auto Loader
# ==========================================

RUTA_EMPLEADOS = "/Volumes/proyecto_final/default/vol_landing/empleados"

ARCHIVO = f"{RUTA_EMPLEADOS}/empleados_lote_01_2026-05-31.csv"

df_empleados = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ARCHIVO)
)

print("ESQUEMA DEL ARCHIVO:")
df_empleados.printSchema()

print("TOTAL DE REGISTROS:", df_empleados.count())

display(df_empleados.limit(10))

In [0]:
# ==========================================
# CONFIGURACIÓN AUTO LOADER - EMPLEADOS
# ==========================================

from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import current_timestamp, col

CATALOGO = "proyecto_final"

RUTA_ORIGEN = (
    "/Volumes/proyecto_final/default/vol_landing/empleados"
)

RUTA_CHECKPOINT = (
    "/Volumes/proyecto_final/default/vol_landing/"
    "_checkpoints/empleados"
)

TABLA_DESTINO = f"{CATALOGO}.bronze.empleados"

# Conservamos los campos originales como STRING en Bronze.
ESQUEMA_EMPLEADOS = StructType([
    StructField("id_empleado", StringType(), True),
    StructField("nombre", StringType(), True),
    StructField("dni", StringType(), True),
    StructField("email", StringType(), True),
    StructField("salario", StringType(), True),
    StructField("sucursal_id", StringType(), True),
    StructField("cargo", StringType(), True),
    StructField("tipo_evento", StringType(), True),
    StructField("fecha_evento", StringType(), True),
    StructField("fecha_registro", StringType(), True)
])

print("Origen:", RUTA_ORIGEN)
print("Checkpoint:", RUTA_CHECKPOINT)
print("Destino:", TABLA_DESTINO)
print("Esquema configurado correctamente.")

In [0]:
# ==========================================
# INGESTA INCREMENTAL - AUTO LOADER
# ==========================================

from pyspark.sql.functions import (
    current_timestamp,
    col
)

# Lectura incremental de archivos CSV
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(ESQUEMA_EMPLEADOS)
    .load(RUTA_ORIGEN)
)

# Agregar campos de trazabilidad
df_bronze = (
    df_stream
    .withColumn("archivo_origen", col("_metadata.file_name"))
    .withColumn("fecha_ingesta", current_timestamp())
)

# Escritura incremental en Delta
query = (
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", RUTA_CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TABLA_DESTINO)
)

query.awaitTermination()

print("Ingesta Bronze de Empleados finalizada correctamente.")

In [0]:
%sql
SELECT
    archivo_origen,
    COUNT(*) AS total_registros,
    MIN(fecha_ingesta) AS primera_ingesta,
    MAX(fecha_ingesta) AS ultima_ingesta
FROM proyecto_final.bronze.empleados
GROUP BY archivo_origen
ORDER BY archivo_origen;